In [1]:
include("GodunovMethod.jl")

MUSCL_errors (generic function with 1 method)

In [ ]:
using Plots
using LaTeXStrings

default(:linewidth, 2)
default(:c, "#000000")

function plot_solution_woodward_collela(::Type{T}, TestCase::Int64, N::Integer, birth_nodes::Bool) where T<:GridType

    default(:xlabel, "Координата")
    default(:legend, false)
    default(:xscale, :identity)
    default(:yscale, :identity)
    default(:markersize, 2)
    default(:markerstrokewidth, 0)

    function BCₗ(t::BigFloat, Ws::Vector{PrimVars})
        # Ячейка 1 уже за границей, поэтому отражаем ячейку 2
        return PrimVars(Ws[2].ρ, - Ws[2].u, Ws[2].p)
    end

    function BCᵣ(t::BigFloat, Ws::Vector{PrimVars})
        # Ячейка end уже за границей, поэтому отражаем ячейку end-1
        return PrimVars(Ws[end-1].ρ, - Ws[end-1].u, Ws[end-1].p)
    end
    
    function IC(x::BigFloat)
        if x ≤ BigFloat("0.1")
            return PrimVars(BigFloat("1"), BigFloat("0"), BigFloat(1e3))
        elseif x ≤ BigFloat("0.9")
            return PrimVars(BigFloat("1"), BigFloat("0"), BigFloat(1e-2))
        end
        return PrimVars(BigFloat("1"), BigFloat("0"), BigFloat(1e2))
    end
    
    grid = Grid(T, N, BigFloat("0"), BigFloat("1"))

    config = ProblemConfig(
        γ = BigFloat("1.4"),
        ε_newton = BigFloat("1e-15"),
        ε_pressure = BigFloat("0.002"),
        ε_density = BigFloat("0.0002"),
        α = 4 * maximum(sound_speed.(IC.(cell_centers(grid)), BigFloat("1.4"))) * min_step(grid) / 10,
        CFL = BigFloat("0.1"),
        t_max = BigFloat("0.038"),
        BCₗ = BCₗ,
        BCᵣ = BCᵣ,
        IC = IC,
        refinement_factor = BigFloat("2"),
        min_time_step = BigFloat("1e-5")
    )

    grid_st = Grid(Static, 4 * N, BigFloat("0"), BigFloat("1"))

    W_st = godunov_method(grid_st, config, false)
    ρs = []
    us = []
    ps = []
    Es = []
    for W in W_st.Ws
        e = calculate_energy(W, config.physics.γ)
        push!(ρs, W.ρ)
        push!(us, W.u)
        push!(ps, W.p)
        push!(Es, e)
    end

    cells_st = cell_centers(grid_st)
    p = plot(
        plot(cells_st, ρs, ylabel="Плотность"),
        plot(cells_st, us, ylabel="Скорость"),
        plot(cells_st, ps, ylabel="Давление"),
        plot(cells_st, Es, ylabel="Внутренняя энергия"),
        layout=(2,2),
        margin=5Plots.mm,
        size=(800,600)
    )

    solution = godunov_method(grid, config, birth_nodes)

    println(length(grid.nodes))

    ρs_n = []
    us_n = []
    ps_n = []
    Es_n = []

    for W in solution.Ws
        e = calculate_energy(W, config.physics.γ)
        push!(ρs_n, W.ρ)
        push!(us_n, W.u)
        push!(ps_n, W.p)
        push!(Es_n, e)
    end
    
    cells = cell_centers(grid)
    scatter!(p[1], cells, ρs_n, c="orange")
    scatter!(p[2], cells, us_n, c="orange")
    scatter!(p[3], cells, ps_n, c="orange")
    scatter!(p[4], cells, Es_n, c="orange")

    # savefig(
    #     plot(
    #         cells, 1 ./ grid.ψs,
    #         c="blue",
    #         margin=5Plots.mm,
    #         size=(800,600),
    #         ylabel="Сжатие сетки"
    #     ),
    #     "./images/maxh/$(T)_$(N)_$(birth_nodes)_psi.png"
    # )

    savefig(p, "./images/spec/$(T)_$(N)_$(birth_nodes)_woodward_collela_t=$(Float64(config.t_max)).png")
    return p
end

In [ ]:
display(plot_solution_woodward_collela(Moving, 1, 200, true))

In [ ]:
function plot_solution_shu_osher(::Type{T}, TestCase::Int64, N::Integer, birth_nodes::Bool) where T<:GridType

    default(:xlabel, "Координата")
    default(:legend, false)
    default(:xscale, :identity)
    default(:yscale, :identity)
    default(:markersize, 2)
    default(:markerstrokewidth, 0)

    function BCₗ(t::BigFloat, Ws::Vector{PrimVars})
        return PrimVars(BigFloat("3.857143"), BigFloat("2.629369"), BigFloat("10.33333"))
    end

    function BCᵣ(t::BigFloat, Ws::Vector{PrimVars})
        PrimVars(BigFloat("1") + BigFloat("0.2") * sin(25), BigFloat("0"), BigFloat("1"))
    end
    
    function IC(x::BigFloat)
        if x ≤ BigFloat("-4")
            return PrimVars(BigFloat("3.857143"), BigFloat("2.629369"), BigFloat("10.33333"))
        end
        return PrimVars(BigFloat("1") + BigFloat("0.2") * sin(5x), BigFloat("0"), BigFloat("1"))
    end
    
    grid = Grid(T, N, BigFloat("-5"), BigFloat("5"))

    config = ProblemConfig(
        γ = BigFloat("1.4"),
        ε_newton = BigFloat("1e-15"),
        ε_pressure = BigFloat("0.002"),
        ε_density = BigFloat("0.0002"),
        α = 4 * maximum(sound_speed.(IC.(cell_centers(grid)), BigFloat("1.4"))) * min_step(grid) / 10,
        CFL = BigFloat("0.4"),
        t_max = BigFloat("1.8"),
        BCₗ = BCₗ,
        BCᵣ = BCᵣ,
        IC = IC,
        refinement_factor = BigFloat("2"),
        min_time_step = BigFloat("1e-5")
    )

    grid_st = Grid(Static, 10 * N, BigFloat("-5"), BigFloat("5"))

    W_st = godunov_method(grid_st, config, false)
    ρs = []
    us = []
    ps = []
    Es = []
    for W in W_st.Ws
        e = calculate_energy(W, config.physics.γ)
        push!(ρs, W.ρ)
        push!(us, W.u)
        push!(ps, W.p)
        push!(Es, e)
    end

    cells_st = cell_centers(grid_st)
    p = plot(
        plot(cells_st, ρs, ylabel="Плотность"),
        plot(cells_st, us, ylabel="Скорость"),
        plot(cells_st, ps, ylabel="Давление"),
        plot(cells_st, Es, ylabel="Внутренняя энергия"),
        layout=(2,2),
        margin=5Plots.mm,
        size=(800,600)
    )

    solution = godunov_method(grid, config, birth_nodes)

    println(length(grid.nodes))

    ρs_n = []
    us_n = []
    ps_n = []
    Es_n = []

    for W in solution.Ws
        e = calculate_energy(W, config.physics.γ)
        push!(ρs_n, W.ρ)
        push!(us_n, W.u)
        push!(ps_n, W.p)
        push!(Es_n, e)
    end
    
    cells = cell_centers(grid)
    plot!(p[1], cells, ρs_n, c="orange", m=:circle, lw=1)
    plot!(p[2], cells, us_n, c="orange", m=:circle, lw=1)
    plot!(p[3], cells, ps_n, c="orange", m=:circle, lw=1)
    plot!(p[4], cells, Es_n, c="orange", m=:circle, lw=1)

    # savefig(
    #     plot(
    #         cells, 1 ./ grid.ψs,
    #         c="blue",
    #         margin=5Plots.mm,
    #         size=(800,600),
    #         ylabel="Сжатие сетки"
    #     ),
    #     "./images/maxh/$(T)_$(N)_$(birth_nodes)_psi.png"
    # )

    savefig(p, "./images/spec/$(T)_$(N)_$(birth_nodes)_shu_osher_t=$(Float64(config.t_max)).png")
    return p
end

In [ ]:
display(plot_solution_shu_osher(Moving, 1, 200, true))

In [ ]:
display(plot_solution_shu_osher(Moving, 1, 200, false))